# 🇰🇷 Hello Clova Agent — 로컬 실행 (WSL2)

HyperCLOVA X SEED Think-14B 기반 Reveal.js 슬라이드 자동 생성 에이전트  
**이 파일은 로컬 워크스테이션(WSL2 Ubuntu) 전용입니다.**

---

## ✅ 셀 실행 순서

| 셀 | 내용 | 소요 시간 |
|----|------|-----------|
| 셀 1 | GPU · 시스템 진단 | ~10초 |
| 셀 2 | CUDA 런타임 설치 (깡통 WSL2 전용) | ~1-5분 |
| 셀 3 | Python 패키지 설치 | ~5-15분 |
| 셀 4 | HuggingFace 로그인 | ~5초 |
| 셀 5 | vLLM 서버 시작 | ~5초 (서버는 백그라운드) |
| 셀 6 | 서버 준비 대기 | **첫 실행 15-30분 (28GB 다운로드)** |
| 셀 7 | 에이전트 UI 실행 | 브라우저에서 localhost:7860 |

---

## 🔧 사전 준비 (노트북 열기 전 1회)

### 1. Windows NVIDIA 드라이버 (GPU가 있는 경우)
- 드라이버 **545 이상** 필요 — WSL2 GPU 지원 최소 버전
- 확인: Windows PowerShell에서 `nvidia-smi`

### 2. WSL2 업데이트 (PowerShell 관리자 권한)
```powershell
wsl --update
wsl --shutdown
```

### 3. HuggingFace 토큰 설정 (WSL2 터미널에서)
```bash
echo 'export HF_TOKEN="hf_xxxxxxxxxxxx"' >> ~/.bashrc
source ~/.bashrc
```
토큰 발급: https://huggingface.co/settings/tokens  
(Read 권한 이상 필요)

### 4. 디스크 여유 공간
모델(28GB) + 임시 파일 = **최소 40GB** 필요  
확인: `df -h ~`

### 5. WSL2 메모리 설정 (기본값이 낮을 경우)
`C:\Users\<username>\.wslconfig` 파일 생성 또는 편집:
```ini
[wsl2]
memory=24GB
swap=8GB
```
변경 후: `wsl --shutdown` → WSL2 재시작


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 1/7  ▌ GPU · 시스템 진단
#
# 이 셀이 실패하면 이후 셀은 의미가 없습니다.
# nvidia-smi 실패 시: 셀 0의 사전 준비 항목 1, 2를 완료했는지 확인하세요.
# ══════════════════════════════════════════════════════════════════════

import subprocess, os, sys, shutil

# ── GPU ───────────────────────────────────────────────────────────────
print("🔍 GPU 진단 중...")
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    print("❌ nvidia-smi 실패 — GPU를 인식하지 못했습니다.")
    print()
    print("해결 방법:")
    print("  1. Windows PowerShell에서 nvidia-smi 동작 확인")
    print("  2. PowerShell (관리자): wsl --update && wsl --shutdown")
    print("  3. WSL2 재시작 후 다시 실행")
    print("  4. Windows NVIDIA 드라이버 545+ 설치 확인")
    raise SystemExit(1)

gpu_name, vram_str, driver_ver, compute = r.stdout.strip().split(",")
vram_gb = int(vram_str.replace(" MiB", "").strip()) / 1024

print(f"  GPU     : {gpu_name.strip()}")
print(f"  VRAM    : {vram_gb:.1f} GB")
print(f"  드라이버: {driver_ver.strip()}")
print(f"  Compute : {compute.strip()}")

# ── Python ────────────────────────────────────────────────────────────
print(f"\n🐍 Python  : {sys.version.split()[0]}")
print(f"   실행경로: {sys.executable}")

if sys.version_info < (3, 10):
    print("⚠️ Python 3.10 이상 권장. 현재 버전에서 문제가 발생할 수 있습니다.")

# ── 디스크 ───────────────────────────────────────────────────────────
home_free_gb = shutil.disk_usage(os.path.expanduser("~")).free / 1e9
print(f"\n💾 디스크 여유: {home_free_gb:.1f} GB (홈 디렉토리 기준)")
if home_free_gb < 40:
    print(f"  ⚠️ 권장 40GB 이상. 현재 {home_free_gb:.1f}GB")
    print("     모델 캐시 기본 경로: ~/.cache/huggingface/")
    print("     변경: export HF_HOME=/mnt/d/hf_cache  # D드라이브 등")

# ── RAM ──────────────────────────────────────────────────────────────
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if "MemAvailable" in line:
                avail_gb = int(line.split()[1]) / 1e6
                print(f"🧠 가용 RAM : {avail_gb:.1f} GB")
                if avail_gb < 12:
                    print("  ⚠️ 12GB 미만 — BitsAndBytes 로딩 중 OOM 위험")
                    print("     .wslconfig 에서 memory= 값 증가 권장 (셀 0 참고)")
                break
except Exception:
    pass

# ── VRAM 기반 전략 결정 ──────────────────────────────────────────────
major, minor = map(int, compute.strip().split("."))
dtype   = "auto" if major >= 8 else "half"   # Ampere(8.0+): bf16 가능
if vram_gb >= 28:
    quant = "none"
    quant_note = "양자화 불필요 — fp16 풀 로딩 (VRAM 충분)"
elif vram_gb >= 14:
    quant = "bitsandbytes"
    quant_note = f"BitsAndBytes 4-bit (~8GB 사용, VRAM {vram_gb:.0f}GB)"
else:
    quant = "bitsandbytes"
    quant_note = f"⚠️ VRAM {vram_gb:.0f}GB 부족 가능 — BitsAndBytes 시도"

print(f"\n📊 로딩 전략:")
print(f"  dtype : {dtype} ({'bf16 지원' if major >= 8 else 'T4급 — float16만 지원'})")
print(f"  quant : {quant_note}")

os.environ["LOCAL_VRAM_GB"] = str(vram_gb)
os.environ["LOCAL_QUANT"]   = quant
os.environ["LOCAL_DTYPE"]   = dtype
print("\n✅ 진단 완료 — 셀 2로 진행하세요")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 2/7  ▌ CUDA 런타임 설치 (깡통 WSL2 전용)
#
# WSL2에서 Windows NVIDIA 드라이버는 GPU 접근(libcuda.so)을 제공하지만
# CUDA 런타임(libcudart.so)은 별도 설치가 필요합니다.
# 이미 CUDA 12.x / 13.x 가 설치된 환경에서는 자동으로 건너뜁니다.
# ══════════════════════════════════════════════════════════════════════

import subprocess, os

def find_cudart():
    r = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True)
    for ver in ["13", "12"]:
        if f"libcudart.so.{ver}" in r.stdout:
            return ver
    return None

cuda_ver = find_cudart()
print(f"🔍 CUDA 런타임: {'libcudart.so.' + cuda_ver + ' ✅' if cuda_ver else '없음 → 설치 필요'}")

if cuda_ver:
    print(f"   CUDA {cuda_ver} 런타임 확인 — 이 셀을 건너뜁니다.")
else:
    print("   NVIDIA apt 저장소 추가 후 CUDA 12 런타임 설치 중...")
    print("   (약 1-3분 소요)\n")

    steps = [
        # apt 저장소 설치 가능 상태로 만들기
        (["apt-get", "update", "-qq"], "apt 업데이트"),
        (["apt-get", "install", "-y", "-q", "gnupg", "software-properties-common",
          "wget"], "빌드 도구"),
        # NVIDIA CUDA keyring (Ubuntu 22.04 x86_64 기준)
        (["bash", "-c",
          "wget -qO /tmp/cuda-keyring.deb "
          "https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb "
          "&& dpkg -i /tmp/cuda-keyring.deb"],
         "NVIDIA apt 키링 등록"),
        (["apt-get", "update", "-qq"], "저장소 업데이트"),
        # CUDA 런타임만 설치 (드라이버는 Windows 측에 있으므로 제외)
        (["apt-get", "install", "-y", "-q", "cuda-cudart-12-8"], "CUDA 12.8 런타임"),
    ]

    failed = False
    for cmd, label in steps:
        print(f"  [{label}]...", end=" ", flush=True)
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print("❌")
            print(f"     명령: {' '.join(cmd)}")
            print(f"     오류: {(r.stderr or r.stdout)[-300:]}")
            failed = True
            break
        print("✅")

    if failed:
        print()
        print("자동 설치 실패. WSL2 터미널에서 수동 실행:")
        print("  sudo apt-get update")
        print("  sudo apt-get install -y gnupg wget software-properties-common")
        print("  wget -O /tmp/cuda-keyring.deb \\")
        print("    https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb")
        print("  sudo dpkg -i /tmp/cuda-keyring.deb")
        print("  sudo apt-get update")
        print("  sudo apt-get install -y cuda-cudart-12-8")
        raise RuntimeError("CUDA 런타임 설치 실패 — 위 명령을 수동으로 실행하세요")

    subprocess.run(["ldconfig"], capture_output=True)
    cuda_ver = find_cudart()
    if cuda_ver:
        print(f"\n✅ CUDA {cuda_ver} 런타임 설치 완료")
    else:
        print("\n⚠️ ldconfig 미인식 — 계속 진행 (vLLM import 시 확인됩니다)")

# ── LD_LIBRARY_PATH 업데이트 ─────────────────────────────────────────
CUDA_PATHS = [
    "/usr/local/cuda-12.8/lib64",
    "/usr/local/cuda-13.0/lib64",
    "/usr/local/cuda/lib64",
    "/usr/lib/wsl/lib",              # WSL2 GPU 드라이버 경로
]
ld = os.environ.get("LD_LIBRARY_PATH", "")
added = []
for p in CUDA_PATHS:
    if os.path.exists(p) and p not in ld:
        ld = f"{p}:{ld}"
        added.append(p)
os.environ["LD_LIBRARY_PATH"] = ld
if added:
    print(f"🔗 LD_LIBRARY_PATH += {', '.join(added)}")

print("\n✅ CUDA 환경 준비 완료 — 셀 3으로 진행하세요")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 3/7  ▌ Python 패키지 설치
#
# pip가 없는 경우 먼저 WSL2 터미널에서 실행:
#   sudo apt-get install -y python3-pip
#
# 설치 순서 중요: transformers를 vllm보다 먼저 설치해야 합니다.
# (HCX 모델은 transformers >= 5.9.0 필요)
# ══════════════════════════════════════════════════════════════════════

import subprocess, sys, importlib, os

# ── pip 확인 ─────────────────────────────────────────────────────────
r = subprocess.run([sys.executable, "-m", "pip", "--version"],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(
        "pip 없음. WSL2 터미널에서 실행:\n"
        "  sudo apt-get install -y python3-pip"
    )
print(f"🔍 pip {r.stdout.split()[1]}\n")

# ── 패키지 목록 (순서 유지) ──────────────────────────────────────────
PACKAGES = [
    # (pip 이름, import 이름, 설명)
    ("transformers>=5.9.0", "transformers", "HCX 모델 호환 버전 (먼저 설치)"),
    ("vllm",               "vllm",         "추론 서버"),
    ("bitsandbytes",       "bitsandbytes", "4-bit 양자화"),
    ("huggingface_hub",    "huggingface_hub", "모델 다운로드 인증"),
    ("langgraph",          "langgraph",    "에이전트 파이프라인"),
    ("openai",             "openai",       "vLLM OpenAI 호환 클라이언트"),
    ("gradio",             "gradio",       "웹 UI"),
    ("python-dotenv",      "dotenv",       ".env 파일 지원"),
]

print("📦 패키지 설치 중...")
all_ok = True
for pip_name, _, desc in PACKAGES:
    print(f"  {pip_name} ({desc})...", end=" ", flush=True)
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", pip_name, "-q"],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        print("✅")
    else:
        print("❌")
        print(f"     {(r.stderr or r.stdout)[-200:]}")
        all_ok = False

if not all_ok:
    raise RuntimeError("일부 패키지 설치 실패 — 위 오류 확인 후 재실행")

# ── import 검증 ──────────────────────────────────────────────────────
print("\n🔍 import 검증:")
fail = False
for _, import_name, _ in PACKAGES:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "ok")
        print(f"  ✅ {import_name} {ver}")
    except ImportError as e:
        print(f"  ❌ {import_name}: {e}")
        fail = True

# vLLM import 심층 검증 (libcudart 링크 오류 조기 발견)
print("\n🔍 vLLM 심층 검증 (libcudart 링크 확인)...")
r = subprocess.run(
    [sys.executable, "-c",
     "from vllm.entrypoints.openai import api_server; print('OK')"],
    capture_output=True, text=True, timeout=60,
    env=os.environ.copy(),
)
if r.returncode != 0:
    err = (r.stderr or r.stdout)
    print(f"❌ vLLM 심층 검증 실패:")
    for line in err.splitlines()[-10:]:
        print(f"   {line}")
    if "libcudart" in err:
        print("\n→ CUDA 런타임 문제. 셀 2를 다시 실행하거나 WSL2를 재시작하세요.")
    fail = True
else:
    print("  ✅ vLLM 심층 검증 통과")

if fail:
    raise RuntimeError("검증 실패 — 위 오류를 해결 후 재실행")

# ── 환경변수 설정 ────────────────────────────────────────────────────
HCX_MODEL = "naver-hyperclovax/HyperCLOVAX-SEED-Think-14B"
os.environ["LLM_API_BASE"] = "http://localhost:8000/v1"
os.environ["LLM_API_KEY"]  = "EMPTY"
os.environ["LLM_MODEL"]    = HCX_MODEL
print(f"\n✅ 환경변수 설정: LLM_API_BASE={os.environ['LLM_API_BASE']}")
print("✅ 패키지 설치 완료 — 셀 4로 진행하세요")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 4/7  ▌ HuggingFace 로그인
#
# HF_TOKEN 설정 우선순위:
#   1. 환경 변수 HF_TOKEN  (권장 — ~/.bashrc에 영구 설정)
#   2. 프로젝트 루트 .env 파일  (HF_TOKEN=hf_xxx)
#   3. 이 셀 실행 시 직접 입력  (이번 세션만 유효)
#
# 미인증 다운로드: 5-10 MB/s → 28GB 모델 = 40-60분 → 세션 종료 위험
# 인증 후 다운로드: 30-100 MB/s → 5-15분
# ══════════════════════════════════════════════════════════════════════

import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN", "")

# .env 파일 폴백
if not HF_TOKEN:
    try:
        from dotenv import load_dotenv
        env_path = os.path.join(os.getcwd(), ".env")
        if os.path.exists(env_path):
            load_dotenv(env_path)
            HF_TOKEN = os.environ.get("HF_TOKEN", "")
            if HF_TOKEN:
                print(f"✅ .env 파일에서 HF_TOKEN 로드: {env_path}")
    except Exception:
        pass

# 직접 입력 폴백
if not HF_TOKEN:
    print("HF_TOKEN 환경 변수 없음.")
    print()
    print("영구 설정 방법 (WSL2 터미널):")
    print('  echo 'export HF_TOKEN="hf_xxxxxxxxxxxx"' >> ~/.bashrc')
    print("  source ~/.bashrc")
    print()
    print("또는 프로젝트 루트에 .env 파일 생성:")
    print('  echo 'HF_TOKEN=hf_xxxxxxxxxxxx' > .env')
    print()
    HF_TOKEN = getpass.getpass("HF_TOKEN 직접 입력 (이번 세션만): ")

if not HF_TOKEN or not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "유효한 HF_TOKEN 없음.\n"
        "토큰 발급: https://huggingface.co/settings/tokens"
    )

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"\n✅ HuggingFace 로그인 완료 (token: {HF_TOKEN[:8]}...)")
print("✅ 셀 5로 진행하세요")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 5/7  ▌ vLLM 서버 시작 (백그라운드)
#
# 이 셀은 즉시 완료됩니다. 실제 서버 준비는 셀 6에서 대기합니다.
#
# [자동 설정]
#   셀 1에서 감지한 GPU VRAM / Compute에 따라 dtype · 양자화를 선택합니다.
#   RTX 30xx/40xx (Ampere 이상): dtype=auto (bfloat16)
#   GTX/RTX 20xx 이하 (Turing):  dtype=half  (float16)
#   VRAM ≥ 28GB: 양자화 없이 fp16 풀 로딩
#   VRAM 14-28GB: BitsAndBytes 4-bit (VRAM ~8GB 사용)
#
# [모델 캐시]
#   첫 실행: ~/.cache/huggingface/ 에 28GB 다운로드
#   재실행:  캐시에서 직접 로드 (~5분)
#   캐시 경로 변경: export HF_HOME=/mnt/d/hf_cache
# ══════════════════════════════════════════════════════════════════════

import subprocess, os, sys

HCX_MODEL = "naver-hyperclovax/HyperCLOVAX-SEED-Think-14B"

vram_gb = float(os.environ.get("LOCAL_VRAM_GB", "0"))
quant   = os.environ.get("LOCAL_QUANT", "bitsandbytes")
dtype   = os.environ.get("LOCAL_DTYPE", "half")

print(f"🖥️ 서버 설정: dtype={dtype}, quant={quant}, VRAM={vram_gb:.1f}GB")
print(f"📦 모델: {HCX_MODEL}\n")

args = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model",                   HCX_MODEL,
    "--host",                    "0.0.0.0",
    "--port",                    "8000",
    "--dtype",                   dtype,
    "--max-model-len",           "4096",
    "--gpu-memory-utilization",  "0.90",
    "--trust-remote-code",
    "--enforce-eager",
    "--served-model-name",       HCX_MODEL,
]
if quant == "bitsandbytes":
    args += ["--quantization", "bitsandbytes", "--load-format", "bitsandbytes"]

env = os.environ.copy()

vllm_proc = subprocess.Popen(
    args,
    env=env,
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)

os.environ["VLLM_PID"] = str(vllm_proc.pid)
print(f"✅ vLLM 프로세스 시작 (PID: {vllm_proc.pid})")
print(f"   로그: tail -f /tmp/vllm.log  (WSL2 터미널에서 확인 가능)")
print()
print("→ 셀 6을 실행해 서버 준비를 대기하세요")
print("  첫 실행: 28GB 다운로드 포함 15-30분")
print("  재실행:  캐시 로드 약 5분")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 6/7  ▌ vLLM 서버 준비 대기
#
# 프로세스 비정상 종료 시 즉시 로그를 출력합니다.
# 60초마다 최근 로그 30줄을 출력합니다.
# ══════════════════════════════════════════════════════════════════════

import time, urllib.request, subprocess, os

HEALTH_URL = "http://localhost:8000/health"
MAX_WAIT   = 1800   # 30분 (첫 실행 28GB 다운로드 포함)
INTERVAL   = 10
LOG_EVERY  = 60

vllm_pid = int(os.environ.get("VLLM_PID", "0"))

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except (ProcessLookupError, PermissionError):
        return False

def show_log(n=30, label=""):
    r = subprocess.run(["tail", f"-{n}", "/tmp/vllm.log"],
                       capture_output=True, text=True)
    if label:
        print(label)
    print(r.stdout or "  (로그 없음)")

def show_engine_error():
    r = subprocess.run(["grep", "-A", "8", "EngineCore.*ERROR", "/tmp/vllm.log"],
                       capture_output=True, text=True)
    if r.stdout.strip():
        print("── EngineCore 에러 (실제 원인) ──")
        print(r.stdout)

print(f"⏳ vLLM 서버 준비 대기 (최대 {MAX_WAIT//60}분, PID: {vllm_pid})")
print(f"   첫 실행은 28GB 다운로드로 15-30분 소요됩니다.\n")

elapsed      = 0
next_log_at  = LOG_EVERY
server_ready = False

while elapsed < MAX_WAIT:
    if vllm_pid and not pid_alive(vllm_pid):
        print(f"\n❌ vLLM 프로세스 (PID {vllm_pid})가 {elapsed}s에 비정상 종료")
        show_engine_error()
        show_log(n=100, label="── /tmp/vllm.log (최근 100줄) ──")
        print()
        print("주요 원인 및 대처:")
        print("  CUDA 런타임 오류 : 셀 2 재실행 → WSL2 재시작")
        print("  RAM/VRAM OOM    : .wslconfig memory= 증가 후 wsl --shutdown")
        print("  BitsAndBytes 오류: 셀 3 재실행 후 셀 5 재실행")
        break

    try:
        urllib.request.urlopen(HEALTH_URL, timeout=3)
        server_ready = True
        break
    except Exception:
        pass

    print(f"  [{elapsed:4d}s] 대기 중...", end="\r")

    if elapsed >= next_log_at:
        show_log(n=20, label=f"\n── [{elapsed}s] 최근 로그 ──")
        next_log_at += LOG_EVERY

    time.sleep(INTERVAL)
    elapsed += INTERVAL

if server_ready:
    print(f"\n✅ vLLM 서버 준비 완료! ({elapsed//60}분 {elapsed%60}초)")
    print("   → 셀 7을 실행해 에이전트 UI를 시작하세요")
elif not (vllm_pid and not pid_alive(vllm_pid)):
    print(f"\n❌ {MAX_WAIT//60}분 내 서버 미시작")
    show_engine_error()
    show_log(n=50, label="── 최근 50줄 ──")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 7/7  ▌ 에이전트 UI 실행
#
# 실행 후 브라우저에서 아래 주소로 접속:
#   http://localhost:7860
#
# (Colab과 달리 외부 공개 URL 불필요 — 로컬 접속만으로 충분)
# 중지: 이 셀 왼쪽의 ■ 버튼 클릭
# ══════════════════════════════════════════════════════════════════════

import os, sys

# share=False: 로컬 접속 전용, 시작 빠름 (Colab의 share=True 불필요)
os.environ["GRADIO_SHARE"] = "false"

print("🚀 Hello Clova Agent UI 시작 중...")
print("   브라우저에서 http://localhost:7860 접속")
print("   중지: 이 셀의 ■ 버튼\n")

!python ui/app.py
